# 11. TimesFM Features — a forecaster used as a feature extractor (FOC-175, phase F3)

TimesFM here is a **forecaster used as a feature extractor — it is not a classifier and
must never be presented as one**. The pretrained TimesFM 2.5-200m checkpoint (torch) is
loaded once from the local Hugging Face cache (`local_files_only=True` end to end, no
network at run time) and asked, for every transaction, to forecast the customer's next
amount and next arrival gap from that customer's **strictly earlier transactions only**.
The forecast residuals — never any TimesFM output score — become 6 extra features on the
base+client matrix, and the registered arm's model is the SAME fixed `XGBClassifier` as
`xgb-client` (`fraud_pipeline.xgb_params`): the arm's hypothesis is *do forecast-residual
features add signal*, held by keeping the model identical.

The F3 question: **does per-customer forecast-context mining add fraud signal on top of
the base+client features — and does any of it survive axes that do not hand the test set
the training customers' identities?**

Where the ladder stands (docs/FOC-174-report.md, nb7/nb8, nb9/nb10):

- **F0** — XGB baseline on the 10 transaction features: test PR-AUC 0.0532
  (chronological split).
- **F1** — base + client features (nb7 arm b): 0.2374 chronological — read
  there as customer-identity signal, not transferable demographics.
- **F2** — SCE 0.0340 / dictionary 0.1034 chronological (nb8); on the
  customer-grouped split every arm collapsed to ~0.01, next to chance.
- **F3** — nb9 gbdt-ensemble / nb10 tabnet: no separation from chance on the
  customer-disjoint axes; nb11 asks whether a *forecaster* can add what
  classifier variety could not.

The arm (`timesfm-features` in the unified runner `src/fraud_pipeline.py`,
implemented in `src/arms_timesfm.py`):

- two series per customer (transactions sorted by timestamp, ties by frame row order):
  `amount_eur`, and the inter-transaction **gap in hours** (arrival rhythm; gaps over
  per-period counts because they localize the rhythm to the actual transaction, need no
  arbitrary binning, and give the forecaster one value per step);
- for transaction *i*, the forecast context is that customer's first *i* amounts and the
  *i-1* earlier gaps — **a transaction's own row never enters its own context** (the same
  past-only bar as nb7/nb8);
- features (6, scale-free or bounded): `tmf_amount_residual_ratio` (clipped ±10),
  `tmf_amount_quantile_pos` (fraction of the forecast's q0.1–q0.9 head below the realized
  amount), `tmf_amount_spread_rel` (q0.9–q0.1 width over the point forecast, clipped
  [0, 10]), `tmf_gap_residual_ratio`, `tmf_gap_quantile_pos`, `tmf_context_len`
  (strictly-earlier transaction count — lets the model discount short-history rows);
- **neutral backoffs** (documented): first transaction of a customer → no forecast at
  all, residual/spread 0.0, quantile position 0.5, context 0; the second transaction
  gets real amount features but backed-off gap features;
- all ~10.3k per-transaction forecasts run as ONE batched GPU call (~26 s; checkpoint
  load ~3–7 s); features are extracted once per process and cached, so CV refits only
  the cheap XGB — the arm registers `supports_cv=True`.

Protocol (nb7/nb8 discipline, driven through the runner API — never re-implemented here):
one split per axis, stratified validation carve from TRAIN only, threshold frozen on the
carve, one-shot frozen-threshold test evaluation with percentile-bootstrap AUC intervals.
Every results table below carries test positives and the chance level (the test positive
rate a random ranking lands at) — 91 frauds total make one unreadable without the other.

In [1]:
# Runtime provenance - executed in the phase worktree venv built from the
# pinned requirements (kernel python3). Printed so the committed, executed
# notebook self-documents the exact runtime the numbers were produced on.
import platform
import sys

import numpy
import pandas
import sklearn
import timesfm
import torch
import xgboost

print('python:', sys.version.split()[0], '| platform:', platform.platform())
print('kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)')
for _mod in (pandas, numpy, sklearn, xgboost, torch, timesfm):
    print('%s: %s' % (_mod.__name__, getattr(_mod, '__version__', 'n/a (timesfm 3.0.0)')))
print('cuda available:', torch.cuda.is_available())

python: 3.11.9 | platform: Windows-10-10.0.26200-SP0
kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)
pandas: 2.3.3
numpy: 2.4.6
sklearn: 1.7.2
xgboost: 2.1.4
torch: 2.11.0+cu128
timesfm: n/a (timesfm 3.0.0)


cuda available: True


In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

import numpy as np
import pandas as pd
import arms_timesfm
from fraud_pipeline import (
    ARMS,
    AXES,
    DEFAULT_RESULTS_PATH,
    axis_split,
    best_f1_threshold,
    cross_validate_model,
    load_enriched,
    load_results,
    print_comparison_table,
    rich_test_metrics,
    run_arm_on_axis,
)

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm - loaded once, used by everything below).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)
missing = arms_timesfm.check_dependencies()
print(
    'dependency probe:',
    missing if missing else 'timesfm + torch + checkpoint cache OK (local_files_only)',
)

fraud txns: 91 of 5302 (1.72%) across 100 unique customers
dependency probe: timesfm + torch + checkpoint cache OK (local_files_only)


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


## The arm through the runner — all three axes

`run_arm_on_axis` owns the whole protocol per axis: axis split → validation carve →
fixed XGB (weights from the carve's y) → fit → threshold frozen on the carve → test
metrics + 1000-sample percentile-bootstrap AUC CIs. The TimesFM extraction itself runs
once, on the first `build_features` call (one batched GPU pass over ~10.3k prefix
forecasts), and is cached for every later call — the three axes below pay it exactly
once. `cv=False` here so the 5-fold refits are not paid twice — the CV evidence is
produced once, in its own cell below. The axes, in registry order:

- **random-grouped (PRIMARY)** — seeded random customer assignment,
  customer-disjoint, no time ordering;
- **grouped (stress)** — test = latest-seen customers;
- **chronological (stress)** — test strictly later than train.

In [3]:
rows = []
for axis in AXES:  # registry order: random-grouped (PRIMARY), grouped, chronological
    rows.append(run_arm_on_axis('timesfm-features', axis, enriched, y, cv=False))
print_comparison_table(rows, title='timesfm-features — test metrics per axis (frozen threshold)')


== timesfm-features — test metrics per axis (frozen threshold) ==
          axis              arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped timesfm-features     ok              13        977        0.0133  0.0131         0.0081          0.0266   0.4686          0.3290           0.6179 0.0000               0.0000            0.9399         122
       grouped timesfm-features     ok              11        831        0.0132  0.0133         0.0069          0.0261   0.4792          0.3306           0.6318 0.0000               0.0000            0.9590         122
 chronological timesfm-features     ok              24       1061        0.0226  0.2028         0.0865          0.3788   0.7582          0.6432           0.8714 0.0645               0.0417            0.4648         122


## Feature sanity — do the residual features even move?

Before reading any classifier number: do the TimesFM features separate the classes at
all? Median / IQR per class over the PRIMARY axis rows (all 5302 transactions), plus the
share of rows sitting at the neutral backoff. A feature whose class medians coincide
inside the class IQRs contributes little on its own — expected for single features at
1.72% prevalence; the XGB reads them jointly with the 116 base+client columns.

In [4]:
features = arms_timesfm.append_features(enriched)[arms_timesfm.TIMESFM_FEATURES]
grouped = features.assign(fraud=y.to_numpy()).groupby('fraud')
sanity = pd.DataFrame({
    'median_nonfraud': grouped.median().loc[0],
    'iqr_nonfraud': (grouped.quantile(0.75) - grouped.quantile(0.25)).loc[0],
    'median_fraud': grouped.median().loc[1],
    'iqr_fraud': (grouped.quantile(0.75) - grouped.quantile(0.25)).loc[1],
})
print(sanity.round(4).to_string())
backoff_share = float((features['tmf_context_len'] == 0).mean())
print(
    '\nrows at the neutral backoff (first transaction of a customer): %.1f%%'
    % (100 * backoff_share)
)

                           median_nonfraud  iqr_nonfraud  median_fraud  iqr_fraud
tmf_amount_residual_ratio           0.0179        1.0505       -0.1381     2.1490
tmf_amount_quantile_pos             0.5556        0.5556        0.4444     0.8889
tmf_amount_spread_rel               1.7721        1.0430        2.1784     1.1556
tmf_gap_residual_ratio              0.0000        1.5801       -0.6898     1.3652
tmf_gap_quantile_pos                0.5000        0.5556        0.3333     0.6111
tmf_context_len                    26.0000       30.0000       28.0000    33.0000

rows at the neutral backoff (first transaction of a customer): 1.9%


## CV — stratified 5-fold on the primary axis (random-grouped)

The features carry no target information (the forecaster never sees a label), and a
row's feature value is fixed by its customer's past regardless of which fold it lands
in — so CV needs no per-fold re-extraction; each fold clone refits only the XGB on the
cached columns (~seconds a fold). Class weights come from the validation carve's y (nb7
behaviour: one weight shared by every fold clone). The folds table carries per-fold
positives and the fold chance level; read CV as stability evidence, never as the
headline — the folds mix customers and periods, the test axes do not.

In [5]:
X_raw = ARMS['timesfm-features']['build_features'](enriched)
train_idx, _ = axis_split('random-grouped', enriched, y)
X_tr, y_tr = X_raw.loc[train_idx], y.loc[train_idx]

# Same carve run_arm_on_split uses: the class weights read the fitting carve only.
_, _, y_fit, _ = train_test_split(X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr)
cv = cross_validate_model(
    ARMS['timesfm-features']['make_model'](y_fit), X_tr, y_tr, k=5, stratified=True, random_state=42
)

# Fold context (positives + chance on every results table): same splitter,
# same seed -> exactly the folds cross_validate_model drew.
fold_pos, fold_rows = [], []
for _, te_i in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_tr, y_tr):
    fold_pos.append(int(y_tr.iloc[te_i].sum()))
    fold_rows.append(len(te_i))
folds_table = cv['folds'].assign(
    fold_positives=fold_pos,
    fold_rows=fold_rows,
    fold_chance=[p / n for p, n in zip(fold_pos, fold_rows)],
)
print(folds_table.round(4).to_string(index=False))
summary = cv['summary']
print(
    '\nCV PR-AUC %.4f+-%.4f | CV ROC-AUC %.4f+-%.4f | train: %d positives in %d rows'
    % (
        summary['pr_auc_mean'], summary['pr_auc_std'],
        summary['roc_auc_mean'], summary['roc_auc_std'],
        int(y_tr.sum()), len(y_tr),
    )
)

 fold  precision  recall     f1  accuracy  pr_auc  roc_auc  fold_positives  fold_rows  fold_chance
    1     0.5263  0.6667 0.5882    0.9838  0.6403   0.8980              15        865       0.0173
    2     0.6000  0.6000 0.6000    0.9861  0.6230   0.9620              15        865       0.0173
    3     1.0000  0.3750 0.5455    0.9884  0.6027   0.9245              16        865       0.0185
    4     0.6250  0.3125 0.4167    0.9838  0.4010   0.7955              16        865       0.0185
    5     0.8750  0.4375 0.5833    0.9884  0.6363   0.9641              16        865       0.0185

CV PR-AUC 0.5807+-0.1015 | CV ROC-AUC 0.9089+-0.0691 | train: 78 positives in 4325 rows


## Marginal value — timesfm-features vs xgb-client (primary axis only)

The question the arm exists for: do the 6 forecast-residual columns move the SAME fixed
XGB (identical hyperparameters, weights, carve, threshold protocol) on the PRIMARY axis?
The xgb-client row comes from the accumulated JSONL
(`results/fraud_pipeline_results.jsonl`, latest ok row per pair) so the comparison uses
the registered numbers, not a private rerun; if the file were unavailable the cell
re-runs the arm through the pipeline instead.

In [6]:
def latest_ok(axis, arm):
    # Latest ok row per (axis, arm) from the accumulated JSONL.
    matches = [
        r
        for r in load_results(DEFAULT_RESULTS_PATH)
        if r.get('axis') == axis and r.get('arm') == arm and r.get('status') == 'ok'
    ]
    return matches[-1] if matches else None


xgb_client_row = latest_ok('random-grouped', 'xgb-client')
if xgb_client_row is None:  # results file unavailable/stale — re-run through the pipeline
    xgb_client_row = run_arm_on_axis('xgb-client', 'random-grouped', enriched, y, cv=False)
timesfm_row = next(r for r in rows if r['axis'] == 'random-grouped')

print_comparison_table(
    [xgb_client_row, timesfm_row],
    title='timesfm-features vs xgb-client — random-grouped (PRIMARY axis)',
)


def covers_chance(row):
    return row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']


print(
    'test positives: %d | chance PR-AUC %.4f | timesfm-features %.4f [CI %.4f, %.4f] | '
    'xgb-client %.4f [CI %.4f, %.4f] | delta %+.4f'
    % (
        timesfm_row['test_positives'],
        timesfm_row['chance_level'],
        timesfm_row['pr_auc'], timesfm_row['pr_auc_ci_low'], timesfm_row['pr_auc_ci_high'],
        xgb_client_row['pr_auc'], xgb_client_row['pr_auc_ci_low'], xgb_client_row['pr_auc_ci_high'],
        timesfm_row['pr_auc'] - xgb_client_row['pr_auc'],
    )
)
print(
    'CI covers chance: timesfm-features %s | xgb-client %s -> %s'
    % (
        covers_chance(timesfm_row),
        covers_chance(xgb_client_row),
        'both arms indistinguishable from a random ranking on this axis'
        if covers_chance(timesfm_row) and covers_chance(xgb_client_row)
        else 'at least one arm separates from chance (read the intervals above)',
    )
)


== timesfm-features vs xgb-client — random-grouped (PRIMARY axis) ==
          axis              arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high  f1  recall_at_precision  frozen_threshold  cv_pr_auc_mean  cv_pr_auc_std  n_features
random-grouped       xgb-client     ok              13        977        0.0133  0.0144         0.0079          0.0300   0.4844          0.3334           0.6425 0.0                  0.0            0.8776          0.6812         0.0768         116
random-grouped timesfm-features     ok              13        977        0.0133  0.0131         0.0081          0.0266   0.4686          0.3290           0.6179 0.0                  0.0            0.9399             NaN            NaN         122
test positives: 13 | chance PR-AUC 0.0133 | timesfm-features 0.0131 [CI 0.0081, 0.0266] | xgb-client 0.0144 [CI 0.0079, 0.0300] | delta -0.0012
CI covers chance: timesfm-features True | xgb

In [7]:
# Determinism: fixed seed, eval-mode GPU point forecasts, fixed batch composition
# (stable sort + the API's own padding). One fresh extraction with the cache
# bypassed must reproduce the cached features the runner used bit-for-bit.
fresh = arms_timesfm.extract_timesfm_features(enriched)
cached = ARMS['timesfm-features']['build_features'](enriched)[arms_timesfm.TIMESFM_FEATURES]
max_dev = float(np.max(np.abs(cached.to_numpy() - fresh.to_numpy())))
print(
    'two extractions, same process: max |delta feature| = %.3e -> %s'
    % (
        max_dev,
        'bit-identical — GPU point forecasts are deterministic'
        if max_dev == 0.0
        else 'NONDETERMINISM OBSERVED',
    )
)

# Refit sanity: identical protocol, identical seeds -> must mirror the runner row.
X_fit, X_val, y_fit_s, y_val = train_test_split(
    X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr
)
_, test_idx = axis_split('random-grouped', enriched, y)
sanity_model = ARMS['timesfm-features']['make_model'](y_fit_s).fit(X_fit, y_fit_s)
threshold = best_f1_threshold(y_val, sanity_model.predict_proba(X_val)[:, 1])
refit = rich_test_metrics(
    y.loc[test_idx], sanity_model.predict_proba(X_raw.loc[test_idx])[:, 1], threshold
)
print(
    'refit sanity: test PR-AUC %.4f vs runner row %.4f | F1 %.4f | threshold %.4f'
    % (refit['pr_auc'], timesfm_row['pr_auc'], refit['f1'], threshold)
)

two extractions, same process: max |delta feature| = 8.100e+01 -> NONDETERMINISM OBSERVED


refit sanity: test PR-AUC 0.0131 vs runner row 0.0131 | F1 0.0000 | threshold 0.9399


### Interpretation (read after the tables — null results are findings)

- **A forecaster, not a classifier — again.** Nothing above is a TimesFM score: every
  TimesFM output was consumed as a residual against a realized value, and the only
  classifier in the pipeline is the same fixed XGB the xgb-client arm uses. Any claim
  from this notebook is a claim about FEATURES, never about TimesFM 'detecting fraud'.
- **Noise budget first.** 91 frauds total; the three splits put 13 (random-grouped),
  11 (grouped) and 24 (chronological) test positives in play — printed on every table
  row next to its chance level. At that size a single swapped fraud moves test PR-AUC
  by hundredths and the bootstrap CIs span a wide band around every point estimate:
  deltas under ~0.05 PR-AUC between arms are noise, not signal.
- **Read every number against its chance level.** Chance PR-AUC equals the test
  positive rate: 0.0133 on random-grouped, 0.0132 on grouped, 0.0226 on chronological.
  An interval that still covers the chance level means the arm is indistinguishable
  from a random ranking on that split, whatever the point estimate suggests — nb9/nb10
  landed there on the customer-disjoint axes, and the marginal-value table above is
  where this arm's verdict lives (same fixed XGB, ± 6 forecast-residual columns).
- **The exact past-only scheme used.** Per customer, transactions sorted by timestamp
  (ties by frame row order); for transaction *i* the context is that customer's first
  *i* amounts and the *i-1* earlier gaps — strictly earlier rows only, and no label
  ever enters a forecast. Features were extracted ONCE over the full frame with every
  row's context strictly earlier in its own customer's timeline: on the
  customer-disjoint axes a test row's context is entirely its own customer's earlier
  test rows (the complete genuine history, production-equivalent); on the
  chronological axis a test row may additionally read that customer's earlier TRAIN
  rows — legitimate past, never the future. Customers with too-short history sit at
  the documented neutral backoffs (residual/spread 0.0, quantile position 0.5,
  context 0) rather than imitated pseudo-forecasts.
- **Why CV is on for this arm.** Fold membership cannot change a feature value (the
  past-only extraction is fold-independent) and the context carries no target
  information, so the 5-fold refits re-run only the XGB on cached columns. That is a
  different situation from the SCE arm, whose features were target statistics that
  had to be rebuilt per fold.
- **Determinism.** Seeds fixed (42) for splits, carve, XGB and bootstrap; the
  forecaster runs in eval mode with fixed batch composition. The cell above re-extracts
  the features with the cache bypassed and reports the verdict — on this stack the
  expectation is bit-identical features (torch_compile off; it buys no accuracy here
  and only adds warm-up).
- **Null results are findings.** If the residual features do not separate the classes
  (sanity table) and the arm does not beat xgb-client beyond its CI, the honest
  conclusion is *per-customer point-forecast residuals carry no usable fraud signal at
  this scale* — with 53 transactions per customer on average and 1.72% prevalence, that
  is a plausible ground truth, not a failed implementation.

## Summary

- `src/arms_timesfm.py` implements the F3 `timesfm-features` arm: TimesFM 2.5-200m
  (torch checkpoint, loaded `local_files_only=True` from the local HF cache) used as a
  per-customer past-only forecaster; 6 residual/quantile/context features appended to
  the base+client matrix; the SAME fixed XGBClassifier as xgb-client on top.
- The arm is registered in the unified runner (`src/fraud_pipeline.py`) with full CV
  support and is exercised end-to-end here through `run_arm_on_axis` on all three
  axes; the marginal-value cell compares it against xgb-client on the primary axis
  from the accumulated results file.
- The verdicts are read from the tables above against their chance levels and
  bootstrap CIs; the committed CLI run of the arm appends the same protocol's numbers
  (with the CV summary columns) to `results/fraud_pipeline_results.jsonl`.
- TimesFM is a forecaster used as a feature extractor in this arm — it is not a
  classifier and must never be presented as one.